In [32]:
import os
import json
import jsonlines
from collections import defaultdict
import time

In [33]:
all_start = time.time()

In [34]:
# CMeIE
# A. c→(h,t)
out_name = 'c_to_ht_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to extract all possible aspect-opinion term pairs from the given text. First, identify potential aspect terms. Then, based on the extracted aspect terms and the given text, extract the corresponding opinion terms.
The output format of the task is: (Aspect Term||Opinion Term) 
Given text: "{text}"
'''
oup_prompt = '{answer_text}'

In [35]:
data_file_list = [
    'processed_train.jsonl',
    'processed_dev.jsonl',
    'processed_test.jsonl',
]
out_file_list = [
    'train.json',
    'dev.json',
    'test.json'
]

In [36]:
read_dir = './ori/14lap/'
out_dir = './14lap/pipeline拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [37]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            inp = inp_prompt.format(text=data['text'])
            spo_list = [(item['aspect_term'],item['opinion_term'],item['sentiment_type']) for item in data['spo_list']]
            processed_spo_list = []
            for spo_item in spo_list:
                if spo_item not in processed_spo_list:
                    processed_spo_list.append(spo_item)
            oup = '\n'.join(['({}||{})'.format(item[0],item[1]) for item in processed_spo_list])
            oup = '```\n' + oup.strip() + '\n```'
            out_data = {
                'instruction':inp,
                'input':'',
                'output':oup,
                'text':data['text'],
                'spo_list':data['spo_list']
            }
            fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:c_to_ht_withtype
cost:0.06秒


In [38]:
# CMeIE
# A. c→(r)
out_name = 'c_to_r_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to identify potential sentiment types based on the given text.
Given the list of sentiment types: ['NEG', 'NEU', 'POS']
The output format of the task is: (Sentiment Type) 
Given text: "{text}"
'''
oup_prompt = '{answer_text}'

In [39]:
read_dir = './ori/14lap/'
out_dir = './14lap/pipeline拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [40]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            inp = inp_prompt.format(text=data['text'])
            spo_list = []
            for item in data['spo_list']:
                if item['sentiment_type'] not in spo_list:
                    spo_list.append(item['sentiment_type'])
            processed_spo_list = []
            for spo_item in spo_list:
                if spo_item not in processed_spo_list:
                    processed_spo_list.append(spo_item)
            oup = '\n'.join(['({})'.format(item) for item in processed_spo_list])
            oup = '```\n' + oup.strip() + '\n```'
            out_data = {
                'instruction':inp,
                'input':'',
                'output':oup,
                'text':data['text'],
                'spo_list':data['spo_list']
            }
            fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:c_to_r_withtype
cost:0.04秒


In [41]:
# CMeIE
# B.  r[s1] c→(h,t)
out_name = 'rc_to_ht_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to extract all aspect-opinion term pairs from the given text and the sentiment type. First, identify potential aspect terms. Then, based on the extracted aspect terms and the sentiment type, extract the corresponding opinion terms.
The output format of the task is: (Aspect Term||Opinion Term) 
Given text: "{text}"
Given sentiment type: "{sentiment_type}"
'''
oup_prompt = '{answer_text}'

In [42]:
read_dir = './ori/14lap/'
out_dir = './14lap/pipeline拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [43]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            relation_to_ht = defaultdict(list)
            for spo_item in data['spo_list']:
                aspect_term = spo_item['aspect_term']
                opinion_term = spo_item['opinion_term']
                sentiment_type = spo_item['sentiment_type']
                relation_to_ht[sentiment_type].append((aspect_term, opinion_term))
            for predicate in relation_to_ht.keys():
                spo_list = relation_to_ht[predicate]
                inp = inp_prompt.format(text=data['text'], sentiment_type = predicate)
                oup = '\n'.join(['({}||{})'.format(item[0].strip(),item[1].strip()) for item in spo_list])
                oup = '```\n' + oup.strip() + '\n```'
                out_data = {
                    'instruction':inp,
                    'input':'',
                    'output':oup,
                    'text':data['text'],
                    'spo_list':data['spo_list']
                }
                fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:rc_to_ht_withtype
cost:0.08秒


In [44]:
# CMeIE
# C. h[s1]t [s2]c→r
out_name = 'htc_to_r_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to identify the sentiment type from from the given aspect-opinion term pair.
Given the list of sentiment types: ['NEG', 'NEU', 'POS']
The input format of the aspect-opinion term pair is: (Aspect Term||Opinion Term) 
The output format of the task is: (Sentiment Type) 
Given text: "{text}"
Given aspect-opinion term pair: ({aspect_term}||{opinion_term})
'''
oup_prompt = '{answer_text}'

In [45]:
read_dir = './ori/14lap/'
out_dir = './14lap/pipeline拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [46]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            spo_def_list = defaultdict(list)
            for spo_item in data['spo_list']:
                aspect_term = spo_item['aspect_term']
                opinion_term = spo_item['opinion_term']
                sentiment_type = spo_item['sentiment_type']
                ht_item = (aspect_term, opinion_term)
                spo_def_list[ht_item].append(sentiment_type)
            for ht_item in spo_def_list.keys():
                predicate = spo_def_list[ht_item]
                inp = inp_prompt.format(text=data['text'], aspect_term = ht_item[0], opinion_term = ht_item[1])
                oup = '\n'.join(['({})'.format(item) for item in predicate])
                oup = '```\n' + oup.strip() + '\n```'
                out_data = {
                    'instruction':inp,
                    'input':'',
                    'output':oup,
                    'text':data['text'],
                    'spo_list':data['spo_list']
                }
                fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:htc_to_r_withtype
cost:0.06秒


In [47]:
all_end = time.time()
all_end-all_start

0.4291527271270752